In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error


In [8]:
df = pd.read_csv("visa_dataset_1000.csv")

print(df.shape)



(1000, 11)


In [9]:
df_encoded = pd.get_dummies(df, drop_first=True)

In [10]:
X = df_encoded.drop("processing_time_days", axis=1)
y = df_encoded["processing_time_days"]


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [12]:
lin_model = LinearRegression()

lin_model.fit(X_train, y_train)

y_pred_lr = lin_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred_lr)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_lr)

print("\nLinear Regression Results")
print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)


Linear Regression Results
MSE: 71.09228799662394
RMSE: 8.431624279854027
MAE: 7.360154522001777


In [13]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print("\nRandom Forest Results")
print("MSE:", mse_rf)
print("RMSE:", rmse_rf)
print("MAE:", mae_rf)



Random Forest Results
MSE: 75.313502
RMSE: 8.678335209013305
MAE: 7.3639


In [14]:
dt_model = DecisionTreeRegressor(random_state=42)

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
mae_dt = mean_absolute_error(y_test, y_pred_dt)

print("\nDecision Tree Results")
print("MSE:", mse_dt)
print("RMSE:", rmse_dt)
print("MAE:", mae_dt)




Decision Tree Results
MSE: 157.72
RMSE: 12.55866234915168
MAE: 10.43


In [16]:
param_grid = {
    "max_depth": [3,5,10,None],
    "min_samples_split": [2,5,10],
    "min_samples_leaf": [1,2,4]
}

grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\nBest Hyperparameters:", grid_search.best_params_)



Best Hyperparameters: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [17]:
best_dt = grid_search.best_estimator_

y_pred_best_dt = best_dt.predict(X_test)

mse_best_dt = mean_squared_error(y_test, y_pred_best_dt)
rmse_best_dt = np.sqrt(mse_best_dt)
mae_best_dt = mean_absolute_error(y_test, y_pred_best_dt)

print("\nTuned Decision Tree Results")
print("MSE:", mse_best_dt)
print("RMSE:", rmse_best_dt)
print("MAE:", mae_best_dt)



Tuned Decision Tree Results
MSE: 73.80013934430491
RMSE: 8.590700748152324
MAE: 7.523893913797727


In [18]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Decision Tree"],
    "MSE": [mse, mse_rf, mse_best_dt],
    "RMSE": [rmse, rmse_rf, rmse_best_dt],
    "MAE": [mae, mae_rf, mae_best_dt]
})

print("\nModel Comparison")
print(results)



Model Comparison
               Model        MSE      RMSE       MAE
0  Linear Regression  71.092288  8.431624  7.360155
1      Random Forest  75.313502  8.678335  7.363900
2      Decision Tree  73.800139  8.590701  7.523894


In [19]:
best_model = results.loc[results["RMSE"].idxmin()]

print("\nBest Model:")
print(best_model)


Best Model:
Model    Linear Regression
MSE              71.092288
RMSE              8.431624
MAE               7.360155
Name: 0, dtype: object
